# Figure 3g — Axoaxonic synapses per glomerulus

Number of axoaxonic synapses per individual synaptic glomerulus, across ten genetically
labelled sensory neuron subtypes in the dSC_APEX volumes.

**Method.** Sensory axons were traced manually in CATMAID, one project per labelled
dataset, and their synapses grouped into glomeruli using an iterative 2 µm distance
threshold (see `data_generation/`, and Methods *Quantification of axoaxonic and
axodendritic synapses*). Each `keyword_counts_<n>_<subtype>.json` holds, as its first
element, the per-glomerulus axoaxonic synapse counts for one subtype. The leading number
fixes the published subtype order.

Violins are ranked by uncapped mean and **capped at 15 for display only** (top tick reads
`15+`); the statistics below use the uncapped values.

**Statistics.** Kruskal-Wallis followed by Dunn post hoc (Holm correction).


In [ ]:
import matplotlib as mpl

# Figure style used for all Figure 3 panels.
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["Helvetica", "Arial", "DejaVu Sans"]
mpl.rcParams["svg.fonttype"] = "none"   # keep text editable in Illustrator
mpl.rcParams["font.size"] = 5


In [ ]:
import glob
import json
import re
from pathlib import Path

import numpy as np

DATA_DIR = Path("../data/catmaid_glomeruli")

files = sorted(
    glob.glob(str(DATA_DIR / "keyword_counts_*_*.json")),
    key=lambda p: int(re.search(r"keyword_counts_(\d+)_", p).group(1)),
)

# Element 0 of each file = per-glomerulus axoaxonic synapse counts.
per_glomerulus = [np.asarray(json.load(open(f))[0], dtype=float) for f in files]

LABELS = [
    "Aβ-RA-LTMR (Ret+)",
    "Aδ-LTMR",
    "C-LTMR",
    "C-HTMR/Heat (MrgprD+)",
    "C-HTMR/Heat (MRGPRA3+)",
    "C-HTMR/Heat (SST+)",
    "A-δ-HTMR (SMR2+_Dorsal)",
    "A-δ-HTMR (SMR2+_Ventral)",
    "C-Heat (SSTR2+)",
    "C-Cold",
]

PALETTE = {
    "Aβ-RA-LTMR (Ret+)":       "#556B2F",
    "Aδ-LTMR":                 "#00A86B",
    "C-LTMR":                  "#BCEBCB",
    "C-HTMR/Heat (MrgprD+)":   "#b23aee",
    "C-HTMR/Heat (MRGPRA3+)":  "#9467bd",
    "C-HTMR/Heat (SST+)":      "#ffbbff",
    "A-δ-HTMR (SMR2+_Dorsal)": "#ff4040",
    "A-δ-HTMR (SMR2+_Ventral)": "#eb5324",
    "C-Heat (SSTR2+)":         "#ffee00",
    "C-Cold":                  "#00bfff",
}

for lab, arr in zip(LABELS, per_glomerulus):
    print(f"{lab:28s} n = {len(arr):4d}  mean = {arr.mean():6.2f}  "
          f"median = {np.median(arr):5.1f}")


In [ ]:
import pandas as pd
import scikit_posthocs as sp
from scipy.stats import kruskal

H, p = kruskal(*per_glomerulus)
print(f"Kruskal-Wallis: H = {H:.4g}, p = {p:.4g}")

long_df = pd.DataFrame([
    {"group": lab, "value": v}
    for lab, arr in zip(LABELS, per_glomerulus) for v in arr
])

dunn = sp.posthoc_dunn(long_df, val_col="value", group_col="group",
                       p_adjust="holm")
dunn


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

Y_CAP = 15          # display cap only; ranking and stats use uncapped values
CM = 1 / 2.54

# rank by uncapped mean, descending
order = sorted(range(len(per_glomerulus)),
               key=lambda i: np.nanmean(per_glomerulus[i]), reverse=True)
data = [np.minimum(per_glomerulus[i], Y_CAP) for i in order]
labs = [LABELS[i] for i in order]
cols = [PALETTE[l] for l in labs]

fig, ax = plt.subplots(figsize=(3 * CM, 2.5 * CM))
x = np.arange(1, len(data) + 1)

parts = ax.violinplot(data, positions=x, widths=0.65,
                      showmeans=False, showmedians=True, showextrema=True)
for body, c in zip(parts["bodies"], cols):
    body.set_facecolor(c)
    body.set_edgecolor("none")
    body.set_linewidth(0)
for k in ["cmedians", "cmins", "cmaxes", "cbars"]:
    parts[k].set_color("black")
    parts[k].set_linewidth(0.5)
parts["cmedians"].set_linewidth(0.8)

rng = np.random.default_rng(0)
for i, arr in enumerate(data):
    ax.scatter(x[i] + rng.uniform(-0.12, 0.12, len(arr)), arr,
               s=0.5, color="black", alpha=0.35, linewidths=0, zorder=3)

ax.set_xlim(0.5, len(data) + 0.5)
ax.set_ylim(-1, 16)
ax.set_xticks(x)
ax.set_xticklabels(labs, rotation=-50, ha="left", fontsize=4)
ax.set_yticks([0, 5, 10, 15])
ax.yaxis.set_major_formatter(FuncFormatter(
    lambda v, _: f"{Y_CAP}+" if np.isclose(v, Y_CAP) else str(int(v))))
ax.set_ylabel("Synapse Count", fontsize=5)
ax.tick_params(axis="x", width=0.5, labelsize=4)
ax.tick_params(axis="y", width=0.5, labelsize=5)
ax.grid(False)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
for s in ("bottom", "left"):
    ax.spines[s].set_linewidth(0.5)

fig.subplots_adjust(left=0.18, right=0.98, bottom=0.30, top=0.98)
fig.savefig("figure3g_axoaxonic_synapses_per_glomerulus.svg", format="svg",
            dpi=300)
plt.show()
